# ContractRisk M3 — prueba controlada de protocolo y modelo

Ejemplos sintéticos: seis respuestas y cuatro decisiones, en tres condiciones (1.5B/v4, 1.5B/v5 y 3B/v5). No ejecuta el gold ni el harness final. Active T4 en una sesión nueva. Los modelos se cargan uno por vez. El candidato 3B requiere una descarga adicional; no se garantiza superioridad. Devuelva todos los resultados y errores. La prueba no evalúa recuperación ni calibra el juez.


In [ ]:
from google.colab import files
from pathlib import Path
import tempfile, zipfile, os, sys, subprocess
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Suba únicamente contractrisk_m3_protocolo_codigo.zip')
archive = next(iter(uploaded))
ROOT = Path(tempfile.mkdtemp(prefix='contractrisk_m3_protocolo_', dir='/content'))
with zipfile.ZipFile(archive) as z:
    for member in z.infolist():
        if not (ROOT / member.filename).resolve().is_relative_to(ROOT.resolve()):
            raise ValueError('Ruta no permitida en ZIP')
    z.extractall(ROOT)
os.chdir(ROOT)
assert Path('m3/run_protocol_probe.py').is_file(), 'ZIP incorrecto'
print(ROOT)


In [ ]:
def run_visible(command, log_path):
    with open(log_path, 'w', encoding='utf-8') as log:
        proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='', flush=True)
            log.write(line)
        code = proc.wait()
    if code:
        raise RuntimeError(f'Comando falló ({code}); consulte {log_path}')

run_visible([sys.executable, '-m', 'pip', 'install', '-r', 'm3/requirements-development-v2.txt'], 'installation.log')
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'm3/tests', '-v'], 'tests.log')
run_visible([sys.executable, '-c', "import sys; sys.path.insert(0,'m3'); from sqlite_support import ensure_sqlite; print(ensure_sqlite()); import torch; assert torch.cuda.is_available(), 'Active GPU T4'; print(torch.cuda.get_device_name(0))"], 'environment.log')


In [ ]:
# La carpeta debe ser nueva: no borre resultados anteriores para repetir.
run_visible([sys.executable, '-u', 'm3/run_protocol_probe.py', '--output', 'protocolo_resultados'], 'execution.log')


In [ ]:
# Ejecute también tras un fallo para conservar diagnóstico.
import json, shutil
result_dir = ROOT / 'protocolo_resultados'
result_dir.mkdir(exist_ok=True)
for log in ROOT.glob('*.log'):
    shutil.copy2(log, result_dir / log.name)
meta = result_dir / 'metadata.json'
print(meta.read_text(encoding='utf-8') if meta.exists() else 'No se creó metadata: revisar logs.')
bundle = shutil.make_archive(str(ROOT / 'contractrisk_m3_protocolo_resultados'), 'zip', result_dir)
files.download(bundle)


Devuelva el ZIP y este notebook descargado con sus salidas. No sustituya las respuestas incorrectas ni los errores. La revisión separará funcionamiento técnico, recuperación, generación y decisiones de herramienta.